# Squares in Squares: Certified Blend + Float64 Rescue

## v1 · every public packing, certified as the doubles Kaggle scores, plus two cheap closures

This notebook builds a full `submission.csv` for all 1,000 puzzles from packings that other participants have published,
and adds two small steps of our own on top of the public merge recipe:

- **float64 rescue.** Kaggle reads `submission.csv` as correctly rounded doubles, so a packing whose contacts clear only in
  digits past a double fails there and the puzzle falls back to a weaker candidate. Instead of dropping such a packing,
  we push every centre away from the box centre by a factor `1 + δ` (δ from 4e-16 to 1e-12) and certify again. The side grows
  by about `L·δ`, far below anything the leaderboard shows.
- **deletion closure.** Removing squares never makes a packing longer. For every `n` we take the accepted packings of
  `n+1 … n+8`, delete the squares that hold the bounding box out, and keep the result if it is shorter than the best packing of `n`.

Everything is judged by the hosts' metric code, unchanged.

| stage | what it does | section |
|---|---|---|
| metric | official `metric/squares-in-squares`, written verbatim to `sis_metric.py` | 1 |
| pool | every submission-shaped CSV in the attached inputs, rounded to doubles, deduplicated | 2 |
| certify | shortest first per `n`, float64 rescue for packings that fail only after rounding | 3 |
| closure | deletion from `n+1 … n+8`, top-down so gains cascade | 4 |
| write | `submission.csv`, re-read as Kaggle reads it and scored once more end to end | 5 |
| analysis | where the score comes from and where the next points are | 6 |

### Credits

The packings are other people's work; this notebook only selects, checks and lightly closes them.

| author | notebook / dataset | used for |
|---|---|---|
| Kaggle (hosts) | [metric/squares-in-squares](https://www.kaggle.com/code/metric/squares-in-squares) | the metric, verbatim (Apache-2.0) |
| Akmal Xodarev (Busya PRIME) | [LB 22.2569880 · Best Public Packings + FQ Hop](https://www.kaggle.com/code/busyaprime/lb-22-2569880-best-public-packings-fq-hop), [FQ quenched packings](https://www.kaggle.com/datasets/busyaprime/squares-in-squares-fq-quenched-packings) | packings; the merge recipe: certify doubles, shortest first, `1e-9` window |
| Nilanjan Saha | [Certified Merge + Projection Quench + GPU Screen](https://www.kaggle.com/code/nilanjansaha123/certified-merge-projection-quench-gpu-screen) | packings; the deletion/border fixpoint idea |
| Malyshev Danil | [Living Atlas of Tight Packings](https://www.kaggle.com/code/malyshevdanil/squares-in-squares-living-atlas-of-tight-packings), [Neighbour Transfer + Squeeze](https://www.kaggle.com/code/malyshevdanil/squares-in-squares-neighbour-transfer-squeeze) | packings; neighbour deletion |
| prvsiyan | [Auto-updater](https://www.kaggle.com/code/prvsiyan/auto-updater), [A Square Deal](https://www.kaggle.com/code/prvsiyan/a-square-deal-research-to-verified-packings) | packings |
| sigmaborov | [Certified Contact Polish](https://www.kaggle.com/code/sigmaborov/squares-in-squares-certified-contact-polish) | packings |
| lucifer19 | [Think · Select · Reclose v24](https://www.kaggle.com/code/lucifer19/think-select-reclose-v24-the-frontier-moves) | packings |
| mufasasimon | [Float64 Trap · Gap Map · Merge](https://www.kaggle.com/code/mufasasimon/squares-in-squares-float64-trap-gap-map-merge) | packings; the float64 round-trip analysis behind the rescue step |
| Amir Hezekiah Espanto | [debut](https://www.kaggle.com/code/amirhezekiahespanto/debut) | packings |
| jazivxt | [Boxed In Sub](https://www.kaggle.com/code/jazivxt/boxed-in-sub), [Boxed In](https://www.kaggle.com/datasets/jazivxt/boxed-in) | packings |
| avazaku | [C-Seam DP Certified Merge](https://www.kaggle.com/code/avazaku/c-seam-dp-certified-merge) | packings |
| alex chilton | [Fault-line DP + Basin Hopping](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2620) | packings |
| Chris Deotte | [FTMC & MILP Refill](https://www.kaggle.com/code/cdeotte/squares-in-squares-ftmc-milp-refill-22-311), [Visual EDA, Metric & Gallery](https://www.kaggle.com/code/cdeotte/squares-in-squares-visual-eda-metric-gallery) | packings; the metric walkthrough |
| Octavio Grau | [Certified Portfolio Blend](https://www.kaggle.com/code/octaviograu/squares-certified-portfolio-blend) | packings |

Each of these notebooks credits further upstream authors (Couzo, Friedman, Ellsworth, Göbel, Trump, Bidwell, Cantrell and many
Kaggle participants); their credit carries through. The analysis in section 6 lists, per puzzle, which source holds the submitted packing.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    WINDOW=1e-9,                      # every candidate within this float side of the best accepted one is certified
    RESCUE_DELTAS=[4e-16, 2e-15, 1e-14, 1e-13, 1e-12],   # centre expansion factors tried after a float64 overlap
    WORKERS=4,                        # Kaggle CPU cores
    CLOSURE_SPAN=8,                   # delete from n+1 ... n+CLOSURE_SPAN
    CLOSURE_BUDGET_S=1800,            # wall-clock cap for the closure stage
    MAX_FILE_MB=400,                  # larger CSVs are skipped
    SKIP_NAMES=(),                    # substrings of input paths to ignore
)

In [ ]:
# ── Session: paths, logging and house style ───────────────────────────────────────────────────
import os, sys, csv, json, math, time, hashlib, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

NB_T0 = time.time()
warnings.filterwarnings("ignore")
INPUT = Path("/kaggle/input")
ROOT = Path("/kaggle/working"); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
COLS = ["n", "square_id", "x", "y", "angle"]
CEIL = {n: math.ceil(math.sqrt(n)) / math.sqrt(n) - 1 for n in range(1, 1001)}   # grid score per puzzle


def log(msg: str) -> None:
    """Print one line prefixed with the elapsed notebook time."""
    print(f"[{time.time() - NB_T0:7.0f}s] {msg}", flush=True)


# House style (as in our other competition notebooks): one accent colour means "this is the point".
INK, MUTED, GRID = "#1f2328", "#8b949e", "#e6e8eb"
ACCENT, SECOND = "#d9480f", "#1c7ed6"
PALETTE = [ACCENT, SECOND, "#495057", "#adb5bd", "#2b8a3e", "#862e9c"]


def apply() -> None:
    """Set matplotlib defaults. Safe to call more than once."""
    mpl.rcParams.update({
        "figure.figsize": (9, 4.6), "figure.dpi": 110, "savefig.dpi": 160,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
        "axes.labelsize": 11, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "legend.frameon": False, "axes.prop_cycle": mpl.cycler(color=PALETTE),
    })


apply()
log(f"inputs mounted: {sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else 'none'}")

## 1. The metric, verbatim

The hosts' metric notebook ([metric/squares-in-squares](https://www.kaggle.com/code/metric/squares-in-squares)) is written to
`sis_metric.py` unchanged and checked by SHA-256. A float SAT test sorts every close pair of squares into separated, overlapping
and undecided; undecided pairs (within `1e-8`) go to `mpmath` interval arithmetic at 50, 100, 200 and 500 digits, and a pair still
undecided at 500 digits fails the puzzle. Every acceptance below is this code's decision.

In [ ]:
%%writefile sis_metric.py
# Verbatim copy of the competition metric, Kaggle notebook metric/squares-in-squares (pulled 2026-11-02). Do not edit.
"""Metric for the "Packing N Unit Squares into the Smallest Square" competition.

Participants submit packings of n unit squares (n=1..1000) inside a bounding
square.  Each square is specified by its center (x, y) and rotation angle.
Coordinates must be plain decimal literals, optionally in scientific notation
(e.g. ``2.70710678118654752440`` or ``1.5e-3``).  Symbolic / closed-form
expressions are not accepted -- submit their decimal expansion instead, at as
many digits as you like.

Scoring
-------
For each puzzle n the bounding side length L is measured
translation-invariantly as ``max(max_x - min_x, max_y - min_y)`` over all
corners.  The per-puzzle score is the excess over the area lower bound

    score(n) = L / sqrt(n) - 1

which is 0.0 exactly when the packing achieves the area lower bound (only
possible for perfect squares packed on the grid) and grows above 0 as the
bounding square wastes area.  **Lower is better**; total score is the sum
across n=1..N.

+-------------------+---------------------------------------------------------+
| Condition         | Score                                                   |
+===================+=========================================================+
| feasible packing  | L / sqrt(n) - 1     (>= 0 always)                       |
+-------------------+---------------------------------------------------------+
| infeasible/invalid| L_0 / sqrt(n) - 1   (grid ceiling — no worse than       |
|                   |                      submitting the trivial grid)       |
+-------------------+---------------------------------------------------------+

For reference: ``L_B = sqrt(n)`` is the area lower bound and
``L_0 = ceil(sqrt(n))`` is the trivial grid packing.  The best-known side
length plays no part -- beating a record just yields a smaller excess -- so
this metric reads no baseline values at all.

Feasibility is checked with interval arithmetic (mpmath) at escalating
precision to certify that no two squares overlap.

Kaggle entry point
------------------
``score(solution, submission, row_id_column_name)`` — called by the Kaggle
backend.  ``solution`` is a DataFrame loaded from ``solution.csv``; only its
``n`` column is read, to fix which puzzles are scored (and which absent ones
are charged the grid ceiling).  ``submission`` is the participant CSV with
columns ``n, square_id, x, y, angle``.

As a side effect it writes ``submission_details.csv``, one row per puzzle with
columns ``n`` and ``L``::

    n,L
    1,1.00000000000000000000000000000000000000000000000000
    2,2.00000000000000000000000000000000000000000000000000
    ...

``L`` is the measured bounding side for puzzle n -- not the per-puzzle score --
written with exactly 50 digits after the decimal point.  Those digits are
significant: L is measured in ``L_DPS``-digit mpmath arithmetic straight from
the submitted decimal strings, not from a double.  Puzzles with no accepted L
(absent from the submission, unparseable, failing column validation, or
rejected as infeasible) are written as zero.
"""

import contextlib
import csv
import io
import math
import multiprocessing
import re
from collections import defaultdict
from collections.abc import Callable, Sequence
from dataclasses import dataclass, field
from fractions import Fraction
from pathlib import Path

import mpmath
import pandas as pd

# ---------------------------------------------------------------------------
# interval
# ---------------------------------------------------------------------------

IVal = mpmath.ctx_iv.ivmpf

# Working precision for the (non-interval) measurement of the bounding side L,
# and the number of digits after the decimal point written for it in
# submission_details.csv.  The margin between the two keeps every written digit
# correct: the only rounding is in the trig of the corner computation, tens of
# digits below the last digit reported.
L_DPS = 60
L_DECIMALS = 50


@contextlib.contextmanager
def working_dps(dps: int):
    """Run a block with both mpmath contexts at ``dps`` digits, then restore them.

    ``mpmath.iv`` carries its own precision, independent of ``mpmath.mp``:
    setting only ``mpmath.mp.dps`` leaves every interval operation at the
    default 53 bits, which caps what a contact can be certified to at ~1e-13 no
    matter how far the escalation sequence climbs.  Both contexts are set here
    because the interval work is interleaved with plain ``mp`` arithmetic.

    >>> with working_dps(500):
    ...     width = mpmath.mpf(mpmath.iv.mpf('0.1').delta)
    >>> width < mpmath.mpf(10) ** -500
    True
    """
    old_iv_dps = mpmath.iv.dps
    old_mp_dps = mpmath.mp.dps
    try:
        mpmath.iv.dps = dps
        mpmath.mp.dps = dps
        yield
    finally:
        mpmath.iv.dps = old_iv_dps
        mpmath.mp.dps = old_mp_dps


def make_interval(value: str, dps: int = 50) -> IVal:
    with working_dps(dps):
        return mpmath.iv.mpf(value)


def iv_cos(x: IVal) -> IVal:
    return mpmath.iv.cos(x)


def iv_sin(x: IVal) -> IVal:
    return mpmath.iv.sin(x)


_ESCALATION_SEQUENCE = [50, 100, 200, 500]


def escalate(
    fn: Callable[[int], bool | None],
    max_dps: int = 500,
    start_dps: int = 50,
) -> bool | None:
    for dps in _ESCALATION_SEQUENCE:
        if dps < start_dps:
            continue
        if dps > max_dps:
            break
        result = fn(dps)
        if result is not None:
            return result
    return None


def iv_le(a: IVal, b: IVal) -> bool | None:
    """Certified less-or-equal: True if a <= b, False if a > b, None if undecided.

    Comparing two overlapping intervals is indeterminate.  mpmath 1.3 reports
    that as None, 1.4 raises ValueError; both mean "undecided" here.
    """
    try:
        return a <= b
    except ValueError:
        return None


def iv_min(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``min``: the tightest interval containing the true minimum.

    Comparison operators are useless for this -- two overlapping intervals have
    no determinate order -- so take it endpoint-wise instead.  min of the lower
    bounds is a lower bound on the true min, and min of the upper bounds is an
    upper bound on it, whichever element turns out to be smallest.
    """
    return mpmath.iv.mpf([min(v.a for v in values), min(v.b for v in values)])


def iv_max(values: Sequence[IVal]) -> IVal:
    """Interval extension of ``max``. See :func:`iv_min`."""
    return mpmath.iv.mpf([max(v.a for v in values), max(v.b for v in values)])


# ---------------------------------------------------------------------------
# parser
# ---------------------------------------------------------------------------

# Bounds the cost of parsing one coordinate.  Far above any useful precision
# (L is reported to 50 decimals), so no honest submission comes near it.
_MAX_LITERAL_CHARS = 100

# Coordinate, angle, and sweep bounds.  For n <= 1000 unit squares, valid
# packings have side L <= 32.  Bounding magnitudes prevents float64/mpmath
# discrepancies and precision loss under extreme translation or rotation.
_MAX_COORD = 1e4
_MAX_ANGLE = 100.0
_AABB_PAD = 1e-7

_DECIMAL_RE = re.compile(r'^[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?$')


def _is_decimal(s: str) -> bool:
    return len(s) <= _MAX_LITERAL_CHARS and bool(_DECIMAL_RE.match(s.strip()))


def _check_decimal(s: str) -> str:
    """Return the stripped literal, or raise if it is not a plain decimal.

    >>> _check_decimal(" 2.70710678118654752440 ")
    '2.70710678118654752440'
    >>> _check_decimal("1.5e-3")
    '1.5e-3'
    >>> _check_decimal("1 + sqrt(2)")
    Traceback (most recent call last):
        ...
    ValueError: not a decimal number: '1 + sqrt(2)'
    """
    s = s.strip()
    if not _is_decimal(s):
        shown = s if len(s) <= 40 else s[:40] + '...'
        raise ValueError(f'not a decimal number: {shown!r}')
    return s


def parse_value(s: str, dps: int = 50) -> IVal:
    """Parse a decimal literal into an enclosing interval at ``dps`` digits."""
    return make_interval(_check_decimal(s), dps)


def parse_value_mp(s: str, dps: int = L_DPS) -> mpmath.mpf:
    """Parse a decimal literal into an ``mpf`` at ``dps`` digits.

    >>> mpmath.nstr(parse_value_mp("2.70710678118654752440"), 21, strip_zeros=False)
    '2.70710678118654752440'
    """
    s = _check_decimal(s)
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        return mpmath.mpf(s)
    finally:
        mpmath.mp.dps = old_dps


def parse_value_float(s: str) -> float:
    """Parse a decimal literal to float.

    Values too large for a double are rejected rather than becoming ``inf``,
    which would poison every downstream geometry check:

    >>> parse_value_float("3.5")
    3.5
    >>> parse_value_float("-2.5")
    -2.5
    >>> parse_value_float("1.5e-3")
    0.0015
    >>> parse_value_float("1e400")
    Traceback (most recent call last):
        ...
    ValueError: value out of range for a double: '1e400'
    """
    s = _check_decimal(s)
    value = float(s)
    if not math.isfinite(value):
        raise ValueError(f'value out of range for a double: {s!r}')
    return value


# ---------------------------------------------------------------------------
# scoring
# ---------------------------------------------------------------------------


def score_puzzle(n: int, s_certified: float) -> float:
    """Score a single puzzle: L / sqrt(n) - 1 (lower is better).

    Result is >= 0 for any feasible packing (equality only for perfect-square
    grid packings).  The best-known side length s* plays no part: beating a
    record simply yields a smaller excess.

    >>> score_puzzle(4, 2.0)
    0.0
    >>> round(score_puzzle(5, 2.707), 6)
    0.210607
    >>> score_puzzle(4, 4.0)
    1.0
    """
    L_B = math.sqrt(n)
    return s_certified / L_B - 1.0


def infeasible_score(n: int) -> float:
    """Penalty score for an infeasible / invalid submission for puzzle n.

    Equal to submitting the trivial grid packing: L_0 / sqrt(n) - 1.
    """
    return math.ceil(math.sqrt(n)) / math.sqrt(n) - 1.0


# ---------------------------------------------------------------------------
# geometry
# ---------------------------------------------------------------------------


@dataclass
class Square:
    x: float
    y: float
    angle: float
    x_str: str = ''
    y_str: str = ''
    angle_str: str = ''
    _corners_float: list[tuple[float, float]] = field(default_factory=list, repr=False)

    def corners_float(self) -> list[tuple[float, float]]:
        if not self._corners_float:
            self._corners_float = compute_corners_float(self.x, self.y, self.angle)
        return self._corners_float


def compute_corners_float(cx: float, cy: float, angle: float) -> list[tuple[float, float]]:
    """Return the four corners of a unit square centered at (cx, cy) rotated by angle.

    >>> corners = compute_corners_float(0.5, 0.5, 0.0)
    >>> [(round(x, 1), round(y, 1)) for x, y in corners]
    [(1.0, 1.0), (0.0, 1.0), (0.0, 0.0), (1.0, 0.0)]
    """
    c = math.cos(angle)
    s = math.sin(angle)
    h = 0.5
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def compute_corners_mp(cx: mpmath.mpf, cy: mpmath.mpf, angle: mpmath.mpf) -> list[tuple[mpmath.mpf, mpmath.mpf]]:
    """``compute_corners_float`` at the ambient ``mpmath.mp`` precision."""
    c = mpmath.cos(angle)
    s = mpmath.sin(angle)
    h = mpmath.mpf(1) / 2
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx + dx1, cy + dy1),
        (cx + dx2, cy + dy2),
        (cx - dx1, cy - dy1),
        (cx - dx2, cy - dy2),
    ]


def measure_side_mp(squares: list['Square'], dps: int = L_DPS) -> mpmath.mpf:
    """Bounding side ``max(max_x - min_x, max_y - min_y)`` at ``dps`` digits.

    Measured from the submitted decimal strings translated relative to Square 0,
    so coordinates carry high precision without loss due to global translation.

    >>> unit = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> measure_side_mp([unit])
    mpf('1.0')
    """
    old_dps = mpmath.mp.dps
    try:
        mpmath.mp.dps = dps
        xs: list[mpmath.mpf] = []
        ys: list[mpmath.mpf] = []
        x0 = parse_value_mp(squares[0].x_str, dps) if squares else mpmath.mpf(0)
        y0 = parse_value_mp(squares[0].y_str, dps) if squares else mpmath.mpf(0)
        for sq in squares:
            corners = compute_corners_mp(
                parse_value_mp(sq.x_str, dps) - x0,
                parse_value_mp(sq.y_str, dps) - y0,
                parse_value_mp(sq.angle_str, dps),
            )
            for x, y in corners:
                xs.append(x)
                ys.append(y)
        return max(max(xs) - min(xs), max(ys) - min(ys))
    finally:
        mpmath.mp.dps = old_dps


def compute_corners_interval(cx_iv: IVal, cy_iv: IVal, angle_iv: IVal) -> list[tuple[IVal, IVal]]:
    c = iv_cos(angle_iv)
    s = iv_sin(angle_iv)
    h = mpmath.iv.mpf('0.5')
    dx1, dy1 = h * c - h * s, h * s + h * c
    dx2, dy2 = -h * c - h * s, -h * s + h * c
    return [
        (cx_iv + dx1, cy_iv + dy1),
        (cx_iv + dx2, cy_iv + dy2),
        (cx_iv - dx1, cy_iv - dy1),
        (cx_iv - dx2, cy_iv - dy2),
    ]


def compute_aabb(
    corners: list[tuple[float, float]],
    pad: float = _AABB_PAD,
) -> tuple[float, float, float, float]:
    xs = [c[0] for c in corners]
    ys = [c[1] for c in corners]
    return (min(xs) - pad, min(ys) - pad, max(xs) + pad, max(ys) + pad)


def aabb_overlap(
    a: tuple[float, float, float, float],
    b: tuple[float, float, float, float],
) -> bool:
    return not (a[2] < b[0] or b[2] < a[0] or a[3] < b[1] or b[3] < a[1])


def _project_corners_float(corners: list[tuple[float, float]], ax: float, ay: float) -> tuple[float, float]:
    projs = [ax * cx + ay * cy for cx, cy in corners]
    return (min(projs), max(projs))


def sat_check_float(sq1: Square, sq2: Square, undecided_tol: float = 1e-8) -> str:
    """Check overlap between two unit squares using the Separating Axis Theorem.

    Returns ``"separated"``, ``"overlapping"``, or ``"undecided"``.

    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(5.0, 5.0, 0.0))
    'separated'
    >>> sat_check_float(Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0))
    'overlapping'
    """
    c1 = sq1.corners_float()
    c2 = sq2.corners_float()

    axes = _get_axes_float(sq1.angle, sq2.angle)

    min_overlap = float('inf')
    for ax, ay in axes:
        min1, max1 = _project_corners_float(c1, ax, ay)
        min2, max2 = _project_corners_float(c2, ax, ay)
        overlap = min(max1, max2) - max(min1, min2)
        if overlap < -undecided_tol:
            return 'separated'
        min_overlap = min(min_overlap, overlap)

    if min_overlap > undecided_tol:
        return 'overlapping'
    return 'undecided'


def _get_axes_float(angle1: float, angle2: float) -> list[tuple[float, float]]:
    c1, s1 = math.cos(angle1), math.sin(angle1)
    c2, s2 = math.cos(angle2), math.sin(angle2)
    return [(c1, s1), (-s1, c1), (c2, s2), (-s2, c2)]


def sat_check_interval(
    sq1_corners: list[tuple[IVal, IVal]],
    sq2_corners: list[tuple[IVal, IVal]],
    angle1_iv: IVal,
    angle2_iv: IVal,
) -> bool | None:
    c1_iv, s1_iv = iv_cos(angle1_iv), iv_sin(angle1_iv)
    c2_iv, s2_iv = iv_cos(angle2_iv), iv_sin(angle2_iv)
    axes = [(c1_iv, s1_iv), (-s1_iv, c1_iv), (c2_iv, s2_iv), (-s2_iv, c2_iv)]

    has_undecided = False
    for ax, ay in axes:
        projs1 = [ax * cx + ay * cy for cx, cy in sq1_corners]
        projs2 = [ax * cx + ay * cy for cx, cy in sq2_corners]

        # Endpoint-wise, via iv_min/iv_max: ordering two overlapping intervals
        # with < is indeterminate, and skipping the projections that compare
        # indeterminately would narrow the span -- which shrinks the overlap and
        # biases the test towards wrongly certifying a pair as separated.
        min1, max1 = iv_min(projs1), iv_max(projs1)
        min2, max2 = iv_min(projs2), iv_max(projs2)

        overlap = iv_min([max1, max2]) - iv_max([min1, min2])
        le_result = iv_le(overlap, mpmath.iv.mpf('0'))
        if le_result is True:
            return True
        if le_result is None:
            has_undecided = True

    if has_undecided:
        return None
    return False


def sort_and_sweep(squares: list[Square]) -> list[tuple[int, int]]:
    n = len(squares)
    if n <= 1:
        return []

    aabbs = []
    for i, sq in enumerate(squares):
        corners = sq.corners_float()
        aabb = compute_aabb(corners)
        aabbs.append((i, aabb))

    aabbs.sort(key=lambda x: x[1][0])

    pairs = []
    for i_idx in range(n):
        i, aabb_i = aabbs[i_idx]
        for j_idx in range(i_idx + 1, n):
            j, aabb_j = aabbs[j_idx]
            if aabb_j[0] > aabb_i[2]:
                break
            if aabb_overlap(aabb_i, aabb_j):
                pairs.append((i, j) if i < j else (j, i))

    return pairs


def verify_puzzle(
    squares: list[Square],
) -> tuple[bool, str]:
    """Check that no two squares overlap, using float SAT then interval arithmetic.

    >>> s1 = Square(0.5, 0.5, 0.0, x_str="0.5", y_str="0.5", angle_str="0")
    >>> s2 = Square(1.5, 0.5, 0.0, x_str="1.5", y_str="0.5", angle_str="0")
    >>> verify_puzzle([s1, s2])
    (True, 'feasible')
    >>> verify_puzzle([Square(0.5, 0.5, 0.0), Square(0.6, 0.6, 0.0)])
    (False, 'Squares 0 and 1 overlap (float check)')
    """
    pairs = sort_and_sweep(squares)
    undecided_pairs = []
    for i, j in pairs:
        result = sat_check_float(squares[i], squares[j])
        if result == 'overlapping':
            return (False, f'Squares {i} and {j} overlap (float check)')
        if result == 'undecided':
            undecided_pairs.append((i, j))

    if undecided_pairs:

        def check_overlap_certified(dps: int) -> bool | None:
            # The whole pair loop runs inside the precision context: parsing at dps
            # digits is not enough, since every later interval operation widens its
            # result to whatever precision mpmath.iv is set to at the time.
            with working_dps(dps):
                still_undecided = []
                for i, j in undecided_pairs:
                    sq_i, sq_j = squares[i], squares[j]
                    cx_i = parse_value(sq_i.x_str, dps)
                    cy_i = parse_value(sq_i.y_str, dps)
                    ai = parse_value(sq_i.angle_str, dps)
                    cx_j = parse_value(sq_j.x_str, dps)
                    cy_j = parse_value(sq_j.y_str, dps)
                    aj = parse_value(sq_j.angle_str, dps)
                    corners_i = compute_corners_interval(cx_i, cy_i, ai)
                    corners_j = compute_corners_interval(cx_j, cy_j, aj)
                    result = sat_check_interval(corners_i, corners_j, ai, aj)
                    if result is False:
                        return False
                    if result is None:
                        still_undecided.append((i, j))
                if still_undecided:
                    return None
                return True

        result = escalate(check_overlap_certified)
        if result is False:
            return (False, 'Squares overlap (certified)')
        if result is None:
            return (False, 'Overlap undecidable at max precision')

    return (True, 'feasible')


# ---------------------------------------------------------------------------
# main / Kaggle entry point
# ---------------------------------------------------------------------------


class ParticipantVisibleError(Exception):
    pass


MAX_N = 1000
KAGGLE_CORES = 4
MIN_PUZZLES_FOR_MP = 4


def _parse_submission_csv(text: str) -> dict[int, list[dict]]:
    reader = csv.DictReader(io.StringIO(text))
    required_cols = {'n', 'square_id', 'x', 'y', 'angle'}
    if not required_cols.issubset(set(reader.fieldnames or [])):
        missing = required_cols - set(reader.fieldnames or [])
        raise ParticipantVisibleError(f'Missing columns: {missing}')

    puzzles: dict[int, list[dict]] = defaultdict(list)
    for row_num, row in enumerate(reader, start=2):
        try:
            n = int(row['n'])
        except (ValueError, TypeError):
            raise ParticipantVisibleError(f'Row {row_num}: invalid n value: {row["n"]!r}')
        if n < 1 or n > MAX_N:
            raise ParticipantVisibleError(f'Row {row_num}: n={n} out of range [1, {MAX_N}]')
        puzzles[n].append(row)

    return dict(puzzles)


def _validate_puzzle(n: int, rows: list[dict]) -> str | None:
    if len(rows) != n:
        return f'n={n}: expected {n} rows, got {len(rows)}'

    square_ids = set()
    for row in rows:
        try:
            sid = int(row['square_id'])
        except (ValueError, TypeError):
            return f'n={n}: invalid square_id: {row["square_id"]!r}'
        square_ids.add(sid)

    expected_ids = set(range(n))
    if square_ids != expected_ids:
        return f'n={n}: square_ids should be 0..{n - 1}, got {sorted(square_ids)}'

    return None


def _build_squares(n: int, rows: list[dict]) -> tuple[list[Square], mpmath.mpf]:
    rows_by_id = {int(r['square_id']): r for r in rows}

    raw_coords = []
    for sid in range(n):
        row = rows_by_id[sid]
        x_str = row['x'].strip()
        y_str = row['y'].strip()
        angle_str = row['angle'].strip()

        try:
            x_f = parse_value_float(x_str)
            y_f = parse_value_float(y_str)
            angle_f = parse_value_float(angle_str)
        except ValueError as e:
            raise ParticipantVisibleError(f'n={n}, square {sid}: cannot parse coordinates: {e}')

        if abs(x_f) > _MAX_COORD or abs(y_f) > _MAX_COORD:
            raise ParticipantVisibleError(f'n={n}, square {sid}: coordinate out of bounds (|x|, |y| must be <= {_MAX_COORD:g})')
        if abs(angle_f) > _MAX_ANGLE:
            raise ParticipantVisibleError(f'n={n}, square {sid}: angle out of bounds (|angle| must be <= {_MAX_ANGLE:g} radians)')

        raw_coords.append((sid, x_str, y_str, angle_str))

    x0_mp = parse_value_mp(raw_coords[0][1], L_DPS)
    y0_mp = parse_value_mp(raw_coords[0][2], L_DPS)
    two_pi = 2 * mpmath.pi

    squares = []
    for sid, x_str, y_str, angle_str in raw_coords:
        x_mp = parse_value_mp(x_str, L_DPS)
        y_mp = parse_value_mp(y_str, L_DPS)
        angle_mp = parse_value_mp(angle_str, L_DPS)

        dx_f = float(x_mp - x0_mp)
        dy_f = float(y_mp - y0_mp)
        angle_reduced_f = float(angle_mp % two_pi)

        squares.append(
            Square(
                x=dx_f,
                y=dy_f,
                angle=angle_reduced_f,
                x_str=x_str,
                y_str=y_str,
                angle_str=angle_str,
            )
        )

    return squares, measure_side_mp(squares)


def evaluate_single_puzzle(n: int, rows: list[dict]) -> tuple[int, float, str | None, str]:
    """Evaluate one puzzle, returning ``(n, score, L, reason)``.

    ``L`` is decimal text with ``L_DECIMALS`` digits after the point, and is
    reported only for a packing that passed the feasibility check: an
    unparseable, malformed, or overlapping puzzle gets None, so no L is
    published for a packing the metric rejected.  Text rather than a float
    because a double cannot hold those digits; the score itself is a double,
    taken from the same high-precision measurement.
    """
    error = _validate_puzzle(n, rows)
    if error:
        return (n, infeasible_score(n), None, error)

    try:
        squares, s_mp = _build_squares(n, rows)
    except ParticipantVisibleError as e:
        return (n, infeasible_score(n), None, str(e))

    feasible, reason = verify_puzzle(squares)
    if not feasible:
        return (n, infeasible_score(n), None, f'infeasible: {reason}')

    L_B = math.sqrt(n)
    if float(s_mp) < L_B - 1e-7:
        return (
            n,
            infeasible_score(n),
            None,
            f'infeasible: L={float(s_mp):.6f} is below area lower bound sqrt({n})={L_B:.6f}',
        )

    puzzle_score = max(0.0, score_puzzle(n, float(s_mp)))
    return (n, puzzle_score, _format_detail(s_mp), 'ok')


def _evaluate_puzzle_worker(args: tuple) -> tuple[int, float, str | None, str]:
    n, rows = args
    try:
        return evaluate_single_puzzle(n, rows)
    except Exception as e:  # noqa: BLE001 - one bad puzzle must not kill the pool
        return (n, infeasible_score(n), None, f'worker error: {e}')


def _evaluate_puzzles_parallel(
    puzzles: dict[int, list[dict]],
    puzzle_ns: set[int],
) -> tuple[float, list[tuple[int, float, str | None, str]]]:
    """Score every n in ``puzzle_ns``; rows for any other n are ignored."""
    work = [(n, rows) for n, rows in sorted(puzzles.items()) if n in puzzle_ns]

    if not work:
        results: list[tuple[int, float, str | None, str]] = []
    elif len(work) < MIN_PUZZLES_FOR_MP:
        results = [evaluate_single_puzzle(n, rows) for n, rows in work]
    else:
        with multiprocessing.Pool(processes=KAGGLE_CORES) as pool:
            results = pool.map(_evaluate_puzzle_worker, work)

    # Any scored n missing from the submission gets the grid-ceiling penalty
    # (lower-is-better metric — otherwise omitting a puzzle would improve the
    # total).
    seen = {n for n, _, _, _ in results}
    for n in puzzle_ns:
        if n not in seen:
            results.append((n, infeasible_score(n), None, 'missing from submission'))
    results.sort(key=lambda r: r[0])

    total = sum(puzzle_score for _, puzzle_score, _, _ in results)
    return total, results


def _format_detail(value: mpmath.mpf) -> str:
    """Decimal text for one per-puzzle L: exactly ``L_DECIMALS`` digits after the point.

    An ``mpf`` is a binary fraction (``man * 2**exp``), so it is converted
    exactly to a ``Fraction`` and rounded once, half-to-even, at the last
    written digit.  ``mpmath.nstr`` counts significant digits rather than
    decimal places, and ``format(mpf, '.50f')`` needs mpmath >= 1.4, so neither
    is usable here.

    >>> _format_detail(mpmath.mpf(2))[:12]
    '2.0000000000'
    >>> len(_format_detail(mpmath.mpf(2)).split('.')[1])
    50
    >>> _format_detail(parse_value_mp('1.41421356237309504880168872420969807856967187537695'))
    '1.41421356237309504880168872420969807856967187537695'
    """
    exact = Fraction(value.man) * Fraction(2) ** value.exp
    scale = 10**L_DECIMALS
    units = round(exact * scale)
    sign = '-' if units < 0 else ''
    whole, frac = divmod(abs(units), scale)
    return f'{sign}{whole}.{frac:0{L_DECIMALS}d}'


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
) -> float:
    """Kaggle entry point. Score a submission against the solution's puzzle set.

    ``solution`` is read for one thing only: the set of n to score.  Everything
    else about a puzzle -- best-known side length, proven-optimal flag, source
    -- is irrelevant to ``L / sqrt(n) - 1``.

    >>> import pandas as pd
    >>> sol = pd.DataFrame({"id": [0], "n": [1]})
    >>> sub = pd.DataFrame({"id": [0], "n": [1], "square_id": [0],
    ...     "x": ["0.5"], "y": ["0.5"], "angle": ["0"]})
    >>> score(sol, sub, "id")  # single unit square, L=1, L/sqrt(1) - 1 = 0.0
    0.0
    """
    solution = solution.drop(columns=[row_id_column_name])
    submission = submission.drop(columns=[row_id_column_name])

    puzzle_ns = {int(n) for n in solution['n']}

    text = submission.to_csv(index=False)
    puzzles = _parse_submission_csv(text)
    total, results = _evaluate_puzzles_parallel(puzzles, puzzle_ns)

    # L is the measured bounding side for puzzle n -- NOT the per-puzzle score
    # L/sqrt(n) - 1.  Puzzles with no accepted L (missing from the submission,
    # unparseable, failing column validation, or rejected as infeasible) keep
    # the historical 0 placeholder: an L measured off an overlapping packing is
    # not a real packing of n squares, and downstream harvesting treats every
    # published L as attainable.  Values arrive here already rendered as decimal
    # text with L_DECIMALS digits after the point; unlike a JSON float they are
    # not re-serialized downstream, so all 50 digits reach the stored file
    # verbatim.
    result_map = {n: L_text for n, _, L_text, _ in results if L_text is not None}
    zero = _format_detail(mpmath.mpf(0))

    output_dir = Path('/kaggle/working')
    if not output_dir.is_dir():
        output_dir = Path('.')
    with open(output_dir / 'submission_details.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'L'])
        for n in range(1, MAX_N + 1):
            writer.writerow([n, result_map.get(n, zero)])

    return float(total)

In [ ]:
# ── Import the metric and check it is the copy we expect ─────────────────────────────────────
import importlib
_body = Path("sis_metric.py").read_text().split("\n", 1)[1].rstrip("\n")   # drop the provenance comment line
assert hashlib.sha256(_body.encode()).hexdigest() == "d269f24b02d5f33fbd2149f76b83414b3b446be4d02776560e04631af3365ccf", "metric copy changed"
sys.path.insert(0, str(ROOT))
import sis_metric as metric
importlib.reload(metric)
log("metric loaded, sha256 d269f24b02d5f33f")

## 2. Every attached packing

Each attached notebook output and dataset is walked for CSV files with the five submission columns (`n, square_id, x, y, angle`).
A puzzle is kept from a file only if it has exactly `n` rows with square ids `0 … n-1`. Every coordinate is turned into the double
Kaggle will score and written back as its shortest round-trip text, so the packing we certify is byte for byte the packing Kaggle reads.
Identical packings from different sources are merged and keep all their sources for credit.

In [ ]:
# ── Collect public packings ───────────────────────────────────────────────
# Idea: busyaprime, LB 22.2569880 · Best Public Packings + FQ Hop.
import csv
import hashlib
import math
import multiprocessing as mp
import time
from collections import defaultdict

import numpy as np


def source_of(path):
    """Identify the notebook, dataset, or top-level input mount."""
    parts = path.relative_to(INPUT).parts
    if len(parts) >= 3 and parts[0] in ('notebooks', 'datasets'):
        return '/'.join(parts[1:3])
    return parts[0]


def packing_L(blob):
    """Measure the float64 bounding side of the square corners."""
    coords = np.array([[float(v) for v in line.split(',')[1:]]
                       for line in blob.splitlines()], dtype=np.float64)
    x, y, a = coords.T
    half = (np.abs(np.cos(a)) + np.abs(np.sin(a))) * 0.5
    return float(max(np.max(x + half) - np.min(x - half),
                     np.max(y + half) - np.min(y - half)))


def read_candidate_file(path):
    """Read one submission-shaped file and return its valid puzzles."""
    groups = defaultdict(list)
    invalid = set()
    try:
        with path.open(newline='', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            if not set(COLS).issubset(reader.fieldnames or []):
                return False, [], 0
            for row in reader:
                try:
                    n = int(row['n'])
                    if not 1 <= n <= 1000:
                        continue
                    sid = int(row['square_id'])
                    values = tuple(float(row[c]) for c in ('x', 'y', 'angle'))
                    if not all(math.isfinite(v) for v in values):
                        raise ValueError('non-finite coordinate')
                    groups[n].append((sid, *values))
                except (ValueError, TypeError, KeyError, OverflowError):
                    try:
                        n = int(row.get('n', ''))
                        if 1 <= n <= 1000:
                            invalid.add(n)
                    except (ValueError, TypeError):
                        pass
    except (OSError, UnicodeError, csv.Error):
        return False, [], 0
    found, skipped = [], 0
    for n, rows in groups.items():
        if n in invalid or len(rows) != n or {r[0] for r in rows} != set(range(n)):
            skipped += 1
            continue
        blob = '\n'.join(','.join((str(sid), *(repr(v) for v in vals)))
                         for sid, *vals in rows)
        found.append((n, blob, packing_L(blob)))
    return True, found, skipped + len(invalid - groups.keys())


_t0 = time.time()
csv.field_size_limit(10**7)
_paths = [p for p in INPUT.rglob('*.csv')
          if p.name.lower() != 'sample_submission.csv'
          and not any(s in str(p) for s in CFG['SKIP_NAMES'])
          and p.stat().st_size <= CFG['MAX_FILE_MB'] * 1024**2]
CANDS = defaultdict(dict)
_shaped = _skipped = 0
with mp.Pool(CFG['WORKERS']) as _pool:
    for _path, (_ok, _found, _bad) in zip(_paths, _pool.imap(read_candidate_file, _paths)):
        _shaped += _ok
        _skipped += _bad
        for _n, _blob, _L in _found:
            _key = hashlib.sha1(_blob.encode()).hexdigest()
            if _key not in CANDS[_n]:
                CANDS[_n][_key] = {'blob': _blob, 'L': _L, 'sources': set(), 'origin': 'public'}
            CANDS[_n][_key]['sources'].add(source_of(_path))
log(f"pool: {len(_paths)} files scanned, {_shaped} shaped, "
    f"{sum(map(len, CANDS.values()))} distinct packings, {_skipped} invalid puzzles, {time.time()-_t0:.1f}s")

## 3. Certify and pick, per n

For each `n` the candidates are certified shortest first (busyaprime's recipe). Every candidate within `1e-9` of the shortest
accepted float side is certified too, and the metric's own score decides among them; ties go to the packing more sources share.

**Float64 rescue.** When the metric rejects a candidate with a *certified* overlap, or an overlap it cannot decide at 500 digits
(contacts that only clear past a double), the centres are pushed away from the box centre by `1 + δ` and the packing is certified
again. The largest δ (`1e-12`) is tried first: if it still overlaps, the overlap is real and the candidate is dropped; otherwise
smaller δ are tried and the smallest that passes is kept. Overlaps the float check already sees (deeper than `1e-8`) are never rescued.

In [ ]:
# ── Certify and select packings ───────────────────────────────────────────
# Idea: busyaprime, LB 22.2569880 · Best Public Packings + FQ Hop.
def certify(n, blob):
    """Certify the exact decimal representations of submitted doubles."""
    rows = []
    for line in blob.splitlines():
        sid, x, y, angle = line.split(',')
        rows.append({'n': str(n), 'square_id': sid, 'x': x, 'y': y, 'angle': angle})
    _, score, L_text, reason = metric.evaluate_single_puzzle(n, rows)
    return reason == 'ok', score, L_text, reason


def rescue(n, blob, delta):
    """Expand square centres around their corner bounding-box centre."""
    rows = [line.split(',') for line in blob.splitlines()]
    coords = np.array([[float(v) for v in row[1:]] for row in rows])
    x, y, a = coords.T
    half = (np.abs(np.cos(a)) + np.abs(np.sin(a))) * 0.5
    cx = (np.min(x - half) + np.max(x + half)) * 0.5
    cy = (np.min(y - half) + np.max(y + half)) * 0.5
    return '\n'.join(f'{sid},{repr(float(cx + (float(sx)-cx)*(1+delta)))},'
                     f'{repr(float(cy + (float(sy)-cy)*(1+delta)))},{angle}'
                     for sid, sx, sy, angle in rows)


def pick(args):
    """Select the best certified candidate and return new rescue candidates."""
    n, candidates = args
    best = None
    best_L = float('inf')
    rejected, added = [], {}
    for key, candidate in sorted(candidates, key=lambda item: item[1]['L']):
        if candidate['L'] > best_L + CFG['WINDOW']:
            break
        ok, score, L_text, reason = certify(n, candidate['blob'])
        if not ok:
            rejected.append((key, reason))
            # Only near-contacts can be rescued: a 'float check' overlap is deeper than 1e-8 and stays an overlap.
            if 'overlap' in reason.lower() and 'float check' not in reason:
                rescued_ok = None
                for delta in sorted(CFG['RESCUE_DELTAS'], reverse=True):   # largest first: if it fails, none will pass
                    blob = rescue(n, candidate['blob'], delta)
                    rkey = hashlib.sha1(blob.encode()).hexdigest()
                    if rkey == key:
                        continue
                    trial = {'blob': blob, 'L': packing_L(blob),
                             'sources': set(candidate['sources']), 'origin': 'rescue'}
                    t_ok, t_score, t_L, t_reason = certify(n, blob)
                    if not t_ok:
                        rejected.append((rkey, t_reason))
                        break                                               # smaller deltas cannot do better
                    rescued_ok = (rkey, trial, t_score, t_L)                # keep the smallest delta that passes
                if rescued_ok:
                    key, candidate, score, L_text = rescued_ok
                    added[key] = candidate
                    ok, reason = True, 'ok'
        if ok:
            best_L = min(best_L, candidate['L'])
            winner = {'key': key, 'score': score, 'L_text': L_text,
                      'origin': candidate['origin'], 'sources': set(candidate['sources'])}
            if best is None or (score, -len(winner['sources'])) < (best['score'], -len(best['sources'])):
                best = winner
    return n, best, rejected, added


def _store_pick(result):
    """Merge a worker result into the shared candidate and result maps."""
    n, winner, rejected, added = result
    for key, candidate in added.items():
        if key in CANDS[n]:
            CANDS[n][key]['sources'].update(candidate['sources'])
        else:
            CANDS[n][key] = candidate
    if winner:
        winner['sources'] = set(CANDS[n][winner['key']]['sources'])
        winner['origin'] = CANDS[n][winner['key']]['origin']
        WIN[n] = winner
    else:
        WIN.pop(n, None)
    REJECTED[n] = rejected


def pick_one(n):
    """Re-certify one puzzle after its candidate pool changes."""
    _store_pick(pick((n, list(CANDS.get(n, {}).items()))))
    return WIN.get(n)


_t0 = time.time()
WIN, REJECTED = {}, {}
with mp.Pool(CFG['WORKERS']) as _pool:
    for _result in _pool.imap_unordered(pick, ((n, list(items.items())) for n, items in CANDS.items()), chunksize=1):
        _store_pick(_result)
_total = sum(WIN[n]['score'] if n in WIN else CEIL[n] for n in range(1, 1001))
log(f"pick: {len(WIN)} accepted, {sum(w['origin'] == 'rescue' for w in WIN.values())} rescue winners, "
    f"total {_total:.12f}, {time.time()-_t0:.1f}s")

## 4. Deletion closure

Removing a square never makes a packing longer, so the accepted packing of any `m > n` is a candidate for `n` once `m - n` squares
are removed (the deletion fixpoint of Nilanjan Saha's and Malyshev Danil's notebooks). Squares are removed greedily: each step drops
the square whose removal leaves the smallest bounding side. `n` runs from 999 down so that gains cascade, `m` from `n+1` to `n+8`,
and every shorter result goes through the same certification as the public packings.

In [ ]:
# ── Deletion closure ───
# Idea: nilanjansaha123 and malyshevdanil, deletion/border fixpoint notebooks.
import hashlib
import time
import numpy as np


def delete_to_n(blob, n):
    """Greedily remove squares minimizing the remaining bounding side."""
    rows = [line.split(',') for line in blob.splitlines()]
    values = np.array([[float(v) for v in row[1:]] for row in rows])
    x, y, angle = values.T
    radius = (np.abs(np.cos(angle)) + np.abs(np.sin(angle))) / 2
    bounds = np.column_stack((x - radius, x + radius, y - radius, y + radius))
    alive = np.arange(len(rows))
    while len(alive) > n:
        b = bounds[alive]
        # For each candidate, only its own contribution to an extreme can change it.
        low = np.min(b[:, (0, 2)], axis=0)
        high = np.max(b[:, (1, 3)], axis=0)
        low2 = np.partition(b[:, (0, 2)], 1, axis=0)[1]
        high2 = np.partition(b[:, (1, 3)], len(b) - 2, axis=0)[-2]
        lo = np.where(b[:, (0, 2)] == low, low2, low)
        hi = np.where(b[:, (1, 3)] == high, high2, high)
        sides = np.max(hi - lo, axis=1)
        axis = int(np.argmax(high - low))
        extreme = np.maximum(high[axis] - b[:, 2 * axis], b[:, 2 * axis + 1] - low[axis])
        remove = np.lexsort((-extreme, sides))[0]
        alive = np.delete(alive, remove)
    b = bounds[alive]
    side = max(np.max(b[:, 1]) - np.min(b[:, 0]), np.max(b[:, 3]) - np.min(b[:, 2]))
    return '\n'.join(f"{sid},{','.join(rows[i][1:])}" for sid, i in enumerate(alive)), float(side)


closure_t0 = time.time()
closure_gains = {}
closure_stopped = False
for n in range(999, 0, -1):
    old_score = WIN[n]['score'] if n in WIN else CEIL[n]
    for m in range(n + 1, min(1000, n + CFG['CLOSURE_SPAN']) + 1):
        if time.time() - closure_t0 >= CFG['CLOSURE_BUDGET_S']:
            closure_stopped = True
            break
        if m not in WIN:
            continue
        parent = CANDS[m][WIN[m]['key']]['blob']
        blob, side = delete_to_n(parent, n)
        current = float(WIN[n]['L_text']) if n in WIN else np.ceil(np.sqrt(n))
        if side >= current - 1e-12:
            continue
        key = hashlib.sha1(blob.encode()).hexdigest()
        candidate = CANDS.setdefault(n, {}).get(key)
        if candidate is None:
            CANDS[n][key] = {'blob': blob, 'L': side, 'sources': {f'closure:{m}'}, 'origin': 'closure'}
        else:
            candidate['sources'].add(f'closure:{m}')
        pick_one(n)
    if n in WIN and WIN[n]['score'] < old_score:
        closure_gains[n] = old_score - WIN[n]['score']
    if closure_stopped:
        break
closure_top = sorted(closure_gains.items(), key=lambda item: -item[1])[:10]
log(f"Deletion closure: {len(closure_gains)} improved, gain {sum(closure_gains.values()):.10g}, top {closure_top}; {'budget reached' if closure_stopped else 'complete'} ({time.time() - closure_t0:.1f}s)")

## 5. Write and score once more

The winners are written to `submission.csv`, read back exactly as Kaggle reads it (`float_precision="round_trip"`), and scored end to end
by the metric's own `score()`. The run stops if that total differs from the sum of the per-puzzle scores by more than `1e-9`.
`per_puzzle.csv` keeps the score, certified side, origin and sources of every puzzle.

In [ ]:
# ── Write and verify submission ───────────────────────────────────────────
import pandas as pd

_t0 = time.time()
_submission = ROOT / 'submission.csv'
_rows = 0
with _submission.open('w', newline='') as _f:
    _writer = csv.writer(_f)
    _writer.writerow(['id', *COLS])
    for _n in sorted(WIN):
        for _line in CANDS[_n][WIN[_n]['key']]['blob'].splitlines():
            _writer.writerow([_rows, _n, *_line.split(',')])
            _rows += 1

_sub = pd.read_csv(_submission, float_precision='round_trip')
_solution = pd.DataFrame({'id': range(1000), 'n': range(1, 1001)})
_total = metric.score(_solution, _sub, 'id')
_expected = sum(WIN[n]['score'] if n in WIN else CEIL[n] for n in range(1, 1001))
assert abs(_total - _expected) < 1e-9, (_total, _expected)
with (ROOT / 'per_puzzle.csv').open('w', newline='') as _f:
    _writer = csv.writer(_f)
    _writer.writerow(['n', 'score', 'L_text', 'origin', 'sources'])
    for _n in range(1, 1001):
        _win = WIN.get(_n)
        _writer.writerow([_n, _win['score'] if _win else CEIL[_n],
                          _win['L_text'] if _win else '', _win['origin'] if _win else '',
                          ';'.join(sorted(_win['sources'])) if _win else ''])
_sha = hashlib.sha256(_submission.read_bytes()).hexdigest()[:16]
log(f"write: sha256 {_sha}, {_rows} rows, {len(WIN)} puzzles, total {_total:.12f}, {time.time()-_t0:.1f}s")

## 6. Where the score comes from

*Figure 1* puts every puzzle's score against the grid, coloured by where the submitted packing came from. *Figure 2* counts, per source,
the puzzles where it holds the submitted packing and where it holds it alone, which is what each source adds that nobody else has.
*Figure 3* shows the saving inside each band between two squares, `u = (n - k²)/(2k + 1)`. The table lists the puzzles with the largest
remaining excess: that is where the next notebook should spend its compute.

In [ ]:
# ── Packing analysis ───
import math
import time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

analysis_t0 = time.time()
ACCENT, SECOND = '#d9480f', '#1c7ed6'
GREY, MID, LIGHT = '#495057', '#adb5bd', '#ced4da'
origin_colors = {'public': SECOND, 'rescue': ACCENT, 'closure': GREY, 'grid': MID}
ns = np.arange(1, 1001)
grid = np.array([CEIL[int(n)] for n in ns])
scores = np.array([WIN[int(n)]['score'] if int(n) in WIN else CEIL[int(n)] for n in ns])
origins = np.array([WIN[int(n)]['origin'] if int(n) in WIN else 'grid' for n in ns])
savings = grid - scores

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ns, grid, color=LIGHT, linewidth=1, label='Grid')
for origin, color in origin_colors.items():
    mask = origins == origin
    if mask.any():
        ax.scatter(ns[mask], scores[mask], s=9, color=color, label=origin, alpha=0.8)
ax.set_yscale('symlog', linthresh=1e-5)
ax.set(title='Score by puzzle', xlabel='n', ylabel='Excess score')
ax.title.set_position((0, 1))
ax.title.set_ha('left')
ax.legend(frameon=False, ncol=4)
plt.tight_layout()
plt.show()
print(f"Takeaway: packings improve on the grid for {int(np.sum(savings > 1e-12))} of 1,000 puzzles, saving {savings.sum():.6f} total score.")

holding, alone = Counter(), Counter()
for winner in WIN.values():
    sources = winner['sources']
    holding.update(sources)
    if len(sources) == 1:
        alone.update(sources)
top_sources = sorted(holding, key=lambda s: (-holding[s], s))[:20]
fig, ax = plt.subplots(figsize=(10, max(3, len(top_sources) * 0.35)))
y = np.arange(len(top_sources))
ax.barh(y, [holding[s] for s in top_sources], color=LIGHT, label='Holds packing')
ax.barh(y, [alone[s] for s in top_sources], color=ACCENT, label='Holds alone')
ax.set_yticks(y, labels=top_sources)
ax.invert_yaxis()
ax.set(title='Sources of submitted packings', xlabel='Puzzles')
ax.title.set_position((0, 1))
ax.title.set_ha('left')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
leader = top_sources[0] if top_sources else None
print(f"Takeaway: {leader} holds {holding[leader]} winning packings, {alone[leader]} alone." if leader else 'Takeaway: no accepted packing sources are available.')

k = np.floor(np.sqrt(ns)).astype(int)
u = (ns - k * k) / (2 * k + 1)
fig, ax = plt.subplots(figsize=(10, 4))
points = ax.scatter(u, savings, c=ns, cmap='viridis', s=10, alpha=0.7)
ax.set(title='Savings across each square-number band', xlabel='Band position', ylabel='Grid score − our score')
ax.title.set_position((0, 1))
ax.title.set_ha('left')
fig.colorbar(points, ax=ax, label='n')
plt.tight_layout()
plt.show()
best = int(np.argmax(savings))
print(f"Takeaway: the largest saving is {savings[best]:.6f} at n={ns[best]}, band position {u[best]:.3f}.")

remaining = sorted(ns, key=lambda n: (-scores[n - 1], n))[:25]
display(pd.DataFrame([{'n': int(n), 'excess score': scores[n - 1], 'origin': origins[n - 1], 'sources': ', '.join(sorted(WIN[int(n)]['sources'])) if int(n) in WIN else 'grid fallback'} for n in remaining]))
log(f"Analysis: 3 figures and top 25 remaining excess scores ({time.time() - analysis_t0:.1f}s)")

## 7. Notes and next steps

- This is a merge of public work plus two cheap closures; the expected public score is close to the best public merge (≈ 22.257).
  The gap to the top of the leaderboard (≈ 22.05) is in packings nobody has published.
- Next: targeted optimisation on the puzzles at the top of the table above (neighbour transfer with contact LP, gadget/fault-line DP for
  large `n`), each in its own notebook, feeding new packings back into this merge as an input.
- If you reuse a packing from here, please credit the source listed for it in `per_puzzle.csv`; the credits table at the top lists every
  notebook and dataset this run reads.